Joins, Datetime operations, Window Functions and File handling in pyspark

In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.appName("joins").getOrCreate()

your 131072x1 screen size is bogus. expect trouble
25/07/25 10:34:49 WARN Utils: Your hostname, IQT-RajenderMudasthu resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/07/25 10:34:49 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/07/25 10:34:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/07/25 10:34:51 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [10]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [90]:
emp_df = spark.read.csv("/home/rajender/pyspark/pyspark/employee.csv", header=True, inferSchema=True)
emp_df.show(3)

+------+---------------+---------------+---------------------+-----------+------+--------+
|emp_id|           name|superior_emp_id|joined_date_timestamp|emp_dept_id|gender|  salary|
+------+---------------+---------------+---------------------+-----------+------+--------+
|     1|           NULL|           NULL|  2016-06-15 04:37:53|        7.0| Other|111107.9|
|     2|Jason Rodriguez|          171.0|  2020-02-13 02:31:07|        1.0|Female|80486.58|
|     3|Justin Peterson|          224.0|  2016-03-18 21:29:35|        7.0|  Male|    NULL|
+------+---------------+---------------+---------------------+-----------+------+--------+
only showing top 3 rows



In [91]:
dep_df = spark.read.csv("/home/rajender/pyspark/pyspark/departments.csv", header=True, inferSchema=True)
dep_df.show(2)

+-------+---------+
|dept_id|dept_name|
+-------+---------+
|      1|       HR|
|      2|  Finance|
+-------+---------+
only showing top 2 rows



In [92]:
emp_df.count()

300

In [93]:
emp_df.printSchema()

root
 |-- emp_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- superior_emp_id: double (nullable = true)
 |-- joined_date_timestamp: timestamp (nullable = true)
 |-- emp_dept_id: double (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: double (nullable = true)



In [94]:
dep_df.printSchema()

root
 |-- dept_id: integer (nullable = true)
 |-- dept_name: string (nullable = true)



In [95]:
# change the datatypes of the emp_id and dept_id 
emp_df = emp_df.withColumn("emp_dept_id",col("emp_dept_id").cast(IntegerType()))
emp_df = emp_df.withColumn("superior_emp_id",col("superior_emp_id").cast(IntegerType()))

In [96]:
emp_df.printSchema()

root
 |-- emp_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- superior_emp_id: integer (nullable = true)
 |-- joined_date_timestamp: timestamp (nullable = true)
 |-- emp_dept_id: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: double (nullable = true)



In [97]:
#INNER JOIN
inner_df = emp_df.join(dep_df, emp_df.emp_dept_id == dep_df.dept_id, "inner")
inner_df.show(3)

+------+---------------+---------------+---------------------+-----------+------+--------+-------+---------+
|emp_id|           name|superior_emp_id|joined_date_timestamp|emp_dept_id|gender|  salary|dept_id|dept_name|
+------+---------------+---------------+---------------------+-----------+------+--------+-------+---------+
|     1|           NULL|           NULL|  2016-06-15 04:37:53|          7| Other|111107.9|      7|  Support|
|     2|Jason Rodriguez|            171|  2020-02-13 02:31:07|          1|Female|80486.58|      1|       HR|
|     3|Justin Peterson|            224|  2016-03-18 21:29:35|          7|  Male|    NULL|      7|  Support|
+------+---------------+---------------+---------------------+-----------+------+--------+-------+---------+
only showing top 3 rows



In [98]:
inner_df.count()

260

In [31]:
# to check the count of nulls in each column in emp_df
total_nulls = emp_df.select([sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in emp_df.columns])
total_nulls.show()

+------+----+---------------+---------------------+-----------+------+------+
|emp_id|name|superior_emp_id|joined_date_timestamp|emp_dept_id|gender|salary|
+------+----+---------------+---------------------+-----------+------+------+
|     0|  12|             40|                   15|         40|    66|    30|
+------+----+---------------+---------------------+-----------+------+------+



In [ ]:
# to check the count of nulls in each column in dep_df
total_nulls_dept = dep_df.select([sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in dep_df.columns])
total_nulls_dept.show()

+-------+---------+
|dept_id|dept_name|
+-------+---------+
|      0|        0|
+-------+---------+



In [32]:
#OUTER JOIN
outer_df = emp_df.join(dep_df, emp_df.emp_dept_id == dep_df.dept_id, "outer")
outer_df.show(3)

+------+------------+---------------+---------------------+-----------+------+--------+-------+---------+
|emp_id|        name|superior_emp_id|joined_date_timestamp|emp_dept_id|gender|  salary|dept_id|dept_name|
+------+------------+---------------+---------------------+-----------+------+--------+-------+---------+
|    25|  Luke Dixon|            132|  2022-09-18 19:49:58|       NULL| Other|50173.48|   NULL|     NULL|
|    26|Steven Flynn|              4|  2017-01-27 12:46:02|       NULL| Other|41008.47|   NULL|     NULL|
|    31|David Miller|            229|  2018-01-05 12:04:57|       NULL| Other|92142.14|   NULL|     NULL|
+------+------------+---------------+---------------------+-----------+------+--------+-------+---------+
only showing top 3 rows



In [33]:
outer_df.count()

300

In [35]:
#Left Join
left_df = emp_df.join(dep_df, emp_df.emp_dept_id == dep_df.dept_id, "left")
left_df.show(3)

+------+---------------+---------------+---------------------+-----------+------+--------+-------+---------+
|emp_id|           name|superior_emp_id|joined_date_timestamp|emp_dept_id|gender|  salary|dept_id|dept_name|
+------+---------------+---------------+---------------------+-----------+------+--------+-------+---------+
|     1|           NULL|           NULL|  2016-06-15 04:37:53|          7| Other|111107.9|      7|  Support|
|     2|Jason Rodriguez|            171|  2020-02-13 02:31:07|          1|Female|80486.58|      1|       HR|
|     3|Justin Peterson|            224|  2016-03-18 21:29:35|          7|  Male|    NULL|      7|  Support|
+------+---------------+---------------+---------------------+-----------+------+--------+-------+---------+
only showing top 3 rows



In [36]:
left_df.count()

300

In [37]:
#left semi join
left_semi_df = emp_df.join(dep_df, emp_df.emp_dept_id == dep_df.dept_id, "leftsemi")
left_semi_df.show(3)

+------+---------------+---------------+---------------------+-----------+------+--------+
|emp_id|           name|superior_emp_id|joined_date_timestamp|emp_dept_id|gender|  salary|
+------+---------------+---------------+---------------------+-----------+------+--------+
|     1|           NULL|           NULL|  2016-06-15 04:37:53|          7| Other|111107.9|
|     2|Jason Rodriguez|            171|  2020-02-13 02:31:07|          1|Female|80486.58|
|     3|Justin Peterson|            224|  2016-03-18 21:29:35|          7|  Male|    NULL|
+------+---------------+---------------+---------------------+-----------+------+--------+
only showing top 3 rows



In [38]:
left_semi_df.count()

260

In [39]:
#left anti join
left_anti_df = emp_df.join(dep_df, emp_df.emp_dept_id == dep_df.dept_id, "leftanti")
left_anti_df.show(3)

+------+------------+---------------+---------------------+-----------+------+--------+
|emp_id|        name|superior_emp_id|joined_date_timestamp|emp_dept_id|gender|  salary|
+------+------------+---------------+---------------------+-----------+------+--------+
|    25|  Luke Dixon|            132|  2022-09-18 19:49:58|       NULL| Other|50173.48|
|    26|Steven Flynn|              4|  2017-01-27 12:46:02|       NULL| Other|41008.47|
|    31|David Miller|            229|  2018-01-05 12:04:57|       NULL| Other|92142.14|
+------+------------+---------------+---------------------+-----------+------+--------+
only showing top 3 rows



In [40]:
left_anti_df.count()

40

In [41]:
#self join
self_df = emp_df.alias("emp1").join(emp_df.alias("emp2"), col("emp1.superior_emp_id") == col("emp2.emp_id"), "inner" )
self_df.show()

+------+------------------+---------------+---------------------+-----------+------+---------+------+--------------------+---------------+---------------------+-----------+------+---------+
|emp_id|              name|superior_emp_id|joined_date_timestamp|emp_dept_id|gender|   salary|emp_id|                name|superior_emp_id|joined_date_timestamp|emp_dept_id|gender|   salary|
+------+------------------+---------------+---------------------+-----------+------+---------+------+--------------------+---------------+---------------------+-----------+------+---------+
|     2|   Jason Rodriguez|            171|  2020-02-13 02:31:07|          1|Female| 80486.58|   171|    Alexandra Tucker|            295|  2017-10-01 12:33:47|          5|  NULL|116232.57|
|     3|   Justin Peterson|            224|  2016-03-18 21:29:35|          7|  Male|     NULL|   224|       Joseph Garcia|            294|  2019-11-23 04:44:51|          5|  NULL|     NULL|
|     4|       George Cook|             23|  2016-

In [ ]:
#to select multiple columns in a dataframe
self_df.select(col("emp1.emp_id"), col("emp1.name"), col("emp1.superior_emp_id").alias("manager_id"), col("emp2.name").alias("manager_name")).show(5)

+------+---------------+----------+----------------+
|emp_id|           name|manager_id|    manager_name|
+------+---------------+----------+----------------+
|     2|Jason Rodriguez|       171|Alexandra Tucker|
|     3|Justin Peterson|       224|   Joseph Garcia|
|     4|    George Cook|        23|        Chad Lee|
|     5|   Taylor Ayala|       136|   Amber Jenkins|
|     6|     Ethan West|       270|   Darin Nichols|
+------+---------------+----------+----------------+
only showing top 5 rows



In [45]:
emp_df.filter(col("emp_id") == 171).show()

+------+----------------+---------------+---------------------+-----------+------+---------+
|emp_id|            name|superior_emp_id|joined_date_timestamp|emp_dept_id|gender|   salary|
+------+----------------+---------------+---------------------+-----------+------+---------+
|   171|Alexandra Tucker|            295|  2017-10-01 12:33:47|          5|  NULL|116232.57|
+------+----------------+---------------+---------------------+-----------+------+---------+



Date Time Operatons

In [47]:
emp_df.show(2)

+------+---------------+---------------+---------------------+-----------+------+--------+
|emp_id|           name|superior_emp_id|joined_date_timestamp|emp_dept_id|gender|  salary|
+------+---------------+---------------+---------------------+-----------+------+--------+
|     1|           NULL|           NULL|  2016-06-15 04:37:53|          7| Other|111107.9|
|     2|Jason Rodriguez|            171|  2020-02-13 02:31:07|          1|Female|80486.58|
+------+---------------+---------------+---------------------+-----------+------+--------+
only showing top 2 rows



In [53]:
#how many employees joined in each year till now.
emp_df=emp_df.withColumn("year", year(col("joined_date_timestamp")))
emp_df.groupBy("year").agg(count("*").alias("Num_employees_joined")).orderBy("year").show()

+----+--------------------+
|year|Num_employees_joined|
+----+--------------------+
|NULL|                  15|
|2015|                  11|
|2016|                  34|
|2017|                  26|
|2018|                  26|
|2019|                  33|
|2020|                  27|
|2021|                  25|
|2022|                  27|
|2023|                  29|
|2024|                  29|
|2025|                  18|
+----+--------------------+



In [54]:
# num of employees joined each month
emp_df.withColumn("month", month("joined_date_timestamp")).groupBy("month").agg(count("*").alias("emp_joined")).orderBy("month").show()

+-----+----------+
|month|emp_joined|
+-----+----------+
| NULL|        15|
|    1|        27|
|    2|        23|
|    3|        29|
|    4|        23|
|    5|        23|
|    6|        22|
|    7|        17|
|    8|        25|
|    9|        18|
|   10|        28|
|   11|        24|
|   12|        26|
+-----+----------+



In [58]:
#which day emp started their work more
emp_df.withColumn("day", date_format(col("joined_date_timestamp"), "EEEE")).groupBy("day").agg(count("*").alias("num_of_emp_joined")).orderBy("day", ascending = False).show()

+---------+-----------------+
|      day|num_of_emp_joined|
+---------+-----------------+
|Wednesday|               46|
|  Tuesday|               44|
| Thursday|               37|
|   Sunday|               47|
| Saturday|               34|
|   Monday|               28|
|   Friday|               49|
|     NULL|               15|
+---------+-----------------+



In [63]:
#convert date to string first
emp_df=emp_df.withColumn("joined_date_timestamp", col("joined_date_timestamp").cast(StringType()))
emp_df.printSchema()

root
 |-- emp_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- superior_emp_id: integer (nullable = true)
 |-- joined_date_timestamp: string (nullable = true)
 |-- emp_dept_id: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: double (nullable = true)
 |-- year: integer (nullable = true)



In [67]:
emp_df = emp_df.withColumn(
    "date_column",
    to_timestamp(col("joined_date_timestamp"), 'yyyy-MM-dd HH:mm:ss')
)

In [69]:
emp_df.show(2)
emp_df.printSchema()

+------+---------------+---------------+---------------------+-----------+------+--------+----+-------------------+
|emp_id|           name|superior_emp_id|joined_date_timestamp|emp_dept_id|gender|  salary|year|        date_column|
+------+---------------+---------------+---------------------+-----------+------+--------+----+-------------------+
|     1|           NULL|           NULL|  2016-06-15 04:37:53|          7| Other|111107.9|2016|2016-06-15 04:37:53|
|     2|Jason Rodriguez|            171|  2020-02-13 02:31:07|          1|Female|80486.58|2020|2020-02-13 02:31:07|
+------+---------------+---------------+---------------------+-----------+------+--------+----+-------------------+
only showing top 2 rows

root
 |-- emp_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- superior_emp_id: integer (nullable = true)
 |-- joined_date_timestamp: string (nullable = true)
 |-- emp_dept_id: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: dou

In [74]:
#how many days have passed from joined date to till now
emp_df.withColumn("no_of_days_passed_from_joining", date_diff(current_timestamp(), emp_df.date_column)).orderBy(col("no_of_days_passed_from_joining"), ascending = False).show(3)

+------+--------------+---------------+---------------------+-----------+------+---------+----+-------------------+------------------------------+
|emp_id|          name|superior_emp_id|joined_date_timestamp|emp_dept_id|gender|   salary|year|        date_column|no_of_days_passed_from_joining|
+------+--------------+---------------+---------------------+-----------+------+---------+----+-------------------+------------------------------+
|   106|   Katie Smith|             40|  2015-08-15 05:43:47|          5| Other|116882.79|2015|2015-08-15 05:43:47|                          3632|
|   233|Gabrielle Haas|            250|  2015-08-18 01:30:59|          2| Other| 119555.9|2015|2015-08-18 01:30:59|                          3629|
|   100|  Michael Reed|             21|  2015-09-23 02:32:00|          1| Other| 45727.79|2015|2015-09-23 02:32:00|                          3593|
+------+--------------+---------------+---------------------+-----------+------+---------+----+-------------------+---

In [77]:
#find the standard deviation of salary 
emp_df.select(stddev("salary").alias("salary_samp_std")).show()

+-----------------+
|  salary_samp_std|
+-----------------+
|26555.55850760862|
+-----------------+



In [78]:
emp_df.select(stddev_pop("salary").alias("salary_population")).show()

+------------------+
| salary_population|
+------------------+
|26506.335928668606|
+------------------+



Window Functions

3 types of window functions -------
I.Ranking functions: row_number(), rank(), dense_rank(), ntile(), percent_rank() ------
II.Analytical Functions: lead(), lag(), cume_dist() --------
III.Aggregate functions: sum(), last(), first(), mean(), stddev(), max()

In [75]:
from pyspark.sql.window import Window

In [99]:
inner_df.show(2)

+------+---------------+---------------+---------------------+-----------+------+--------+-------+---------+
|emp_id|           name|superior_emp_id|joined_date_timestamp|emp_dept_id|gender|  salary|dept_id|dept_name|
+------+---------------+---------------+---------------------+-----------+------+--------+-------+---------+
|     1|           NULL|           NULL|  2016-06-15 04:37:53|          7| Other|111107.9|      7|  Support|
|     2|Jason Rodriguez|            171|  2020-02-13 02:31:07|          1|Female|80486.58|      1|       HR|
+------+---------------+---------------+---------------------+-----------+------+--------+-------+---------+
only showing top 2 rows



In [101]:
window_spec1 = Window.partitionBy("dept_name").orderBy(desc("salary"))
inner_df = inner_df.withColumn("rownum", row_number().over(window_spec1))
inner_df.show(truncate=False)

+------+----------------+---------------+---------------------+-----------+------+---------+-------+-----------+------+
|emp_id|name            |superior_emp_id|joined_date_timestamp|emp_dept_id|gender|salary   |dept_id|dept_name  |rownum|
+------+----------------+---------------+---------------------+-----------+------+---------+-------+-----------+------+
|47    |Taylor Taylor   |NULL           |NULL                 |3          |Female|114326.4 |3      |Engineering|1     |
|298   |Stacey Chen     |217            |2025-02-11 00:48:00  |3          |NULL  |113949.24|3      |Engineering|2     |
|33    |Patricia Carter |274            |NULL                 |3          |Other |110237.94|3      |Engineering|3     |
|200   |Patricia Carr   |54             |2020-10-27 11:30:20  |3          |Male  |109933.46|3      |Engineering|4     |
|257   |Gary Nielsen    |6              |2020-11-03 15:57:10  |3          |Other |108009.56|3      |Engineering|5     |
|237   |James Barnes    |NULL           

In [102]:
#dense rank function
window_spec2 = Window.partitionBy("dept_name").orderBy(desc("salary"))
inner_df = inner_df.withColumn("denserank_col", dense_rank().over(window_spec2))
inner_df.show(2)

+------+-------------+---------------+---------------------+-----------+------+---------+-------+-----------+------+-------------+
|emp_id|         name|superior_emp_id|joined_date_timestamp|emp_dept_id|gender|   salary|dept_id|  dept_name|rownum|denserank_col|
+------+-------------+---------------+---------------------+-----------+------+---------+-------+-----------+------+-------------+
|    47|Taylor Taylor|           NULL|                 NULL|          3|Female| 114326.4|      3|Engineering|     1|            1|
|   298|  Stacey Chen|            217|  2025-02-11 00:48:00|          3|  NULL|113949.24|      3|Engineering|     2|            2|
+------+-------------+---------------+---------------------+-----------+------+---------+-------+-----------+------+-------------+
only showing top 2 rows



In [104]:
#Analytical functions cume_dist, lead, lag
window_spec3 = Window.partitionBy("dept_name").orderBy(desc("salary"))
inner_df = inner_df.withColumn("cum_dist_column", cume_dist().over(window_spec3))
inner_df.show()

+------+----------------+---------------+---------------------+-----------+------+---------+-------+-----------+------+-------------+--------------------+
|emp_id|            name|superior_emp_id|joined_date_timestamp|emp_dept_id|gender|   salary|dept_id|  dept_name|rownum|denserank_col|     cum_dist_column|
+------+----------------+---------------+---------------------+-----------+------+---------+-------+-----------+------+-------------+--------------------+
|    47|   Taylor Taylor|           NULL|                 NULL|          3|Female| 114326.4|      3|Engineering|     1|            1|0.023809523809523808|
|   298|     Stacey Chen|            217|  2025-02-11 00:48:00|          3|  NULL|113949.24|      3|Engineering|     2|            2|0.047619047619047616|
|    33| Patricia Carter|            274|                 NULL|          3| Other|110237.94|      3|Engineering|     3|            3| 0.07142857142857142|
|   200|   Patricia Carr|             54|  2020-10-27 11:30:20|       

In [105]:
#lead example
window_spec4 = Window.partitionBy("dept_name").orderBy(desc("salary"))

lead(col_name, Number_of_offset_rows, default_value_if_no_such_row_exists)


In [106]:
inner_df = inner_df.withColumn("lead_column", lead("salary", 1).over(window_spec4))
inner_df.show()

+------+----------------+---------------+---------------------+-----------+------+---------+-------+-----------+------+-------------+--------------------+-----------+
|emp_id|            name|superior_emp_id|joined_date_timestamp|emp_dept_id|gender|   salary|dept_id|  dept_name|rownum|denserank_col|     cum_dist_column|lead_column|
+------+----------------+---------------+---------------------+-----------+------+---------+-------+-----------+------+-------------+--------------------+-----------+
|    47|   Taylor Taylor|           NULL|                 NULL|          3|Female| 114326.4|      3|Engineering|     1|            1|0.023809523809523808|  113949.24|
|   298|     Stacey Chen|            217|  2025-02-11 00:48:00|          3|  NULL|113949.24|      3|Engineering|     2|            2|0.047619047619047616|  110237.94|
|    33| Patricia Carter|            274|                 NULL|          3| Other|110237.94|      3|Engineering|     3|            3| 0.07142857142857142|  109933.46

In [107]:
#
inner_df = inner_df.withColumn("lag_column", lag("salary", 3).over(window_spec4))
inner_df.show(10)

+------+---------------+---------------+---------------------+-----------+------+---------+-------+-----------+------+-------------+--------------------+-----------+----------+
|emp_id|           name|superior_emp_id|joined_date_timestamp|emp_dept_id|gender|   salary|dept_id|  dept_name|rownum|denserank_col|     cum_dist_column|lead_column|lag_column|
+------+---------------+---------------+---------------------+-----------+------+---------+-------+-----------+------+-------------+--------------------+-----------+----------+
|    47|  Taylor Taylor|           NULL|                 NULL|          3|Female| 114326.4|      3|Engineering|     1|            1|0.023809523809523808|  113949.24|      NULL|
|   298|    Stacey Chen|            217|  2025-02-11 00:48:00|          3|  NULL|113949.24|      3|Engineering|     2|            2|0.047619047619047616|  110237.94|      NULL|
|    33|Patricia Carter|            274|                 NULL|          3| Other|110237.94|      3|Engineering|    

In [108]:
#to delete specific columns in dataframe
self_df.printSchema()

root
 |-- emp_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- superior_emp_id: integer (nullable = true)
 |-- joined_date_timestamp: timestamp (nullable = true)
 |-- emp_dept_id: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: double (nullable = true)
 |-- emp_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- superior_emp_id: integer (nullable = true)
 |-- joined_date_timestamp: timestamp (nullable = true)
 |-- emp_dept_id: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: double (nullable = true)



In [109]:
self_df = self_df.drop("joined_date_timestamp", "gender", "emp_id")
self_df.printSchema()

root
 |-- name: string (nullable = true)
 |-- superior_emp_id: integer (nullable = true)
 |-- emp_dept_id: integer (nullable = true)
 |-- salary: double (nullable = true)
 |-- name: string (nullable = true)
 |-- superior_emp_id: integer (nullable = true)
 |-- emp_dept_id: integer (nullable = true)
 |-- salary: double (nullable = true)



In [ ]:
| Area             | API/Library              | Example Function                            |
| ---------------- | ------------------------ | ------------------------------------------- |
| RDDs             | `pyspark.RDD`            | `map()`, `filter()`                         |
| DataFrames       | `pyspark.sql`            | `select()`, `groupBy()`                     |
| SQL Functions    | `pyspark.sql.functions`  | `col()`, `avg()`, `when()`                  |
| Data Types       | `pyspark.sql.types`      | `StructType`, `StringType`                  |
| Window Functions | `pyspark.sql.window`     | `row_number()`, `rank()`                    |
| Machine Learning | `pyspark.ml`             | `LogisticRegression()`, `VectorAssembler()` |
| Streaming        | `pyspark.sql.streaming`  | `readStream`, `writeStream`                 |
| Graph Processing | `graphframes` (external) | `bfs()`, `shortestPaths()`                  |


In [110]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

spark = SparkSession.builder.appName("DataFrameExample").getOrCreate()

# Define schema
# schema is a collection of structfields in a list of a structtype
schema = StructType([
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True)
])

# Sample data
data = [("Alice", 30), ("Bob", 28)]

# Create DataFrame
df = spark.createDataFrame(data, schema)

df.show()

25/07/25 22:34:12 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


+-----+---+
| name|age|
+-----+---+
|Alice| 30|
|  Bob| 28|
+-----+---+



25/07/30 10:02:58 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 15598143 ms exceeds timeout 120000 ms
25/07/30 10:02:59 WARN SparkContext: Killing executors is not supported by current scheduler.
25/07/30 10:03:02 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint

In [ ]:
## DATASET EXAMPLE 
## this is applicable in scala or java only

case class Person(name: String, age: Int)
val ds = spark.createDataset(Seq(Person("Alice", 30), Person("Bob", 28)))
ds.show()